# Scrapy
- 웹사이트에서 데이터 수집을 위한 오픈소스 파이썬 프레임워크
- 멀티스레딩으로 데이터 수집
- daum news 상품데이터 수집

In [4]:
# install scrapy
# !pip install scrapy

In [3]:
import scrapy, requests
from scrapy.http import TextResponse

## 1. make project

In [8]:
!scrapy startproject news

New Scrapy project 'news', using template directory 'C:\Users\User\anaconda3\Lib\site-packages\scrapy\templates\project', created in:
    C:\Users\User\python\0919_Wep Crawling\news

You can start your first spider with:
    cd news
    scrapy genspider example example.com


In [5]:
# 디렉토리의 구조를 보여주는 것이다.
!tree news /F

폴더 PATH의 목록입니다.
볼륨 일련 번호는 BEA2-02D7입니다.
C:\USERS\USER\PYTHON\0919_WEPCRAWLING\NEWS
│  scrapy.cfg
│  
└─news
    │  items.py
    │  middlewares.py
    │  pipelines.py
    │  settings.py
    │  __init__.py
    │  
    ├─spiders
    │  │  spider.py
    │  │  __init__.py
    │  │  
    │  └─__pycache__
    │          __init__.cpython-312.pyc
    │          
    └─__pycache__
            settings.cpython-312.pyc
            __init__.cpython-312.pyc
            


- scrapy structure
    - items : 데이터의 모양 정의
    - middewares : 수집할때 header 정보와 같은 내용을 설정
    - pipelines : 데이터를 수집한 후에 코드를 실행
    - settings : robots.txt 규칙, 크롤링 시간 텀등을 설정
    - spiders : 크롤링 절차를 정의

## 2. xpath
BeautifulSoup 대신에 Xpath 사용
- link, contents

### xpath

- html element 선택하는 방법
- scrapy 에서는 기본적으로 xpath를 사용
- syntax
    - // : 최상위 엘리먼트
    - \* : 모든 하위 엘리먼트 : css selector의 한칸띄우기와 같다.
    - [@id="value"] : 속성값 선택
    - / : 한단계 하위 엘리먼트 : css selector의 >와 같다.
    - \[n]:nth-child(n)

In [7]:
import scrapy, requests
from scrapy.http import TextResponse

In [9]:
url = 'https://news.daum.net/'
response = requests.get(url)
response = TextResponse(response.url, body=response.text, encoding='utf-8')
response

<200 https://news.daum.net/>

In [11]:
# 다음 뉴스에서 뉴스 기사 제목 하나 클릭하면 나오는 element.
selector = '/html/body/div[2]/main/section/div/div[1]/div[1]/ul/li/div/div/strong/a/@href'  # @href을 뒤에 붙이면 20240923151053270(li번호..?)까지 가져올 수 있다.
links = response.xpath(selector).extract()  
len(links), links[:3]  # 이전에 selector로 할때는 for문을 사용했었는데 Xpath는 반복문 사용없이 출력해올 수 있음

(20,
 ['https://v.daum.net/v/20240923160652789',
  'https://v.daum.net/v/20240923160138488',
  'https://v.daum.net/v/20240923154254673'])

In [13]:
# 링크데이터 수집 > 링크 데이터 안에 있는 상세 페이지 수집
link = links[0]
response = requests.get(link)
response = TextResponse(response.url, body=response.text, encoding='utf-8')
response

<200 https://v.daum.net/v/20240923160652789>

In [15]:
title = response.xpath('//*[@id="mArticle"]/div[1]/h3/text()')[0].extract()   # 뒤에 '/text()'를 붙여서 텍스트 뽑아오기.
title

'"505호로 메시지를" 엘리베이터에 붙은 메모가 가져온 변화는… [이.단.아]'

## 3. items.py
- Data Model

In [ ]:
# %load news/news/items.py
# Define here the models for your scraped items
#
# See documentation in:
# https://docs.scrapy.org/en/latest/topics/items.html

In [17]:
%%writefile news/news/items.py
import scrapy

class NewsItem(scrapy.Item):
    title = scrapy.Field()
    link = scrapy.Field()

Overwriting news/news/items.py


## 4. spider.py
- wirte crawling process

In [19]:
%%writefile news/news/spiders/spider.py
import scrapy
from news.items import NewsItem

class NewsSpider(scrapy.Spider):
    name = 'news'
    allow_domain = ['daum.net']
    start_urls = ['https://news.daum.net']
    
    def parse(self, response):
        selector = '/html/body/div[2]/main/section/div/div[1]/div[1]/ul/li/div/div/strong/a/@href'
        links = response.xpath(selector).extract()
        for link in links:
            yield scrapy.Request(link, callback=self.parse_content)

    def parse_content(self, response):
        item = NewsItem()
        item['link'] = response.url
        item['title'] = response.xpath('//*[@id="mArticle"]/div[1]/h3/text()')[0].extract()
        yield item

Overwriting news/news/spiders/spider.py


## 5. run scrapy
- news 디렉토리에서 아래의 커멘드 실행
- scrapy crawl news -o news.csv

In [21]:
%pwd

'C:\\Users\\User\\python\\0919_WepCrawling'